# Практика 2: данные и инфраструктура обучения

## 1. Установка и импорты

In [1]:
# Библиотека для загрузки датасетов (с Hugging Face)
!pip install datasets  

  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached pyarrow-24.0.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached multiprocess-0.70.19-py314-none-any.whl.metadata (7.2 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-macosx_10_15_universal2.whl.metadata (40 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
Using cached datasets-4.8.4-py3-none-any.whl (526 kB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
Using cached multiprocess-0.70.19-py314-none-any.whl (160 kB)
Using cached pyarrow-24.0.0-cp314-cp314-macosx_12_0_arm64.whl (35.0 MB)
Using cached requests-2.33.1-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.7-cp314-cp314-macosx_10_15_universal2.whl (309

In [2]:
!pip install transformers


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
!pip install lightning clearml tensorboard

  Using cached clearml-2.1.5-py2.py3-none-any.whl.metadata (18 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached furl-2.1.4-py2.py3-none-any.whl.metadata (25 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached pathlib2-2.3.7.post1-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached pyjwt-2.12.1-py3-none-any.whl.metadata (4.1 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached grpcio-1.80.0-cp314-cp314-macosx_11_0_universal2.whl.metadata (3.8 kB)
  Using cached markdown-3.10.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached protobuf-7.34.1-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached werkzeug-3.1.8-py3-none-any.whl.metadata (

In [4]:
import numpy
import random
import time

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from datasets import load_dataset

from transformers import AutoTokenizer
from transformers import GPT2Config, GPT2LMHeadModel
from transformers import default_data_collator

from lightning import LightningDataModule, LightningModule, Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import TensorBoardLogger

/Users/egorprokopov/Documents/Work/MI/MNNA-2026-Spring/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
SEED = 239

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [6]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device = torch.device(device)
device

device(type='mps')

## 2. Данные

Для обучения языковых моделей нужны данные и чем больше модель - тем больше данных необходимо

На лекции мы рассматривали преимущественно обучение трансформера для решения задачи машинного перевода. Такая задача требует больших корпусов размеченных текстов и размечать текст для такой задачи - очень дорого.

Пойдем по другому пути: будем обучать модель предсказывать следующий токен текстовой последовательности. Это задача авторегрессионной генерации текста. Самое главное преимущество такого обучения --- не требуется разметка данных. Такое обучение называется self supervised learning.


LLM обучают на очень больших корпусах текстов, которые собирают из различных источников:
- веб-страницы
- книги
- статьи
- код
- документация
- диалоги
- внутренние данные компании


### 2.1 Учебный датасет
Возьмем датасет `wikitext-2-raw-v1`, который очистим и подготовим для обучения

In [7]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
dataset

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

Выведем первые объекты

In [10]:
for i in range(1, 5):
    print(f"--- example {i} ---")
    print(repr(dataset["train"][i]["text"]))

--- example 1 ---
' = Valkyria Chronicles III = \n'
--- example 2 ---
''
--- example 3 ---
' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'
--- example 4 ---
" The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also und

Как видно, не все тексты одинаково полезны. Нужно очистить данные, а для этого удалить:
- пустые строки
- короткие отрывки
- заголовки
- 'неизвестные' языки
- дубликаты

Сейчас удалим только простые строки

In [11]:
non_empty_examples = [
    example["text"]
    for example in dataset["train"]
    if len(example["text"].strip()) > 0
]

print("Всего строк в train:", len(dataset["train"]))
print("Непустых строк в train:", len(non_empty_examples))
print()
print(non_empty_examples[0])

Всего строк в train: 36718
Непустых строк в train: 23767

 = Valkyria Chronicles III = 



### 2.2 Потоковая загрузка данных

Если датасет большой, то скачивать его целиком может быть проблематично. В этом случае можно использовать `streaming=True`, чтобы читать данные итеративно

In [12]:
streaming_dataset = load_dataset(
    "wikitext",
    "wikitext-2-raw-v1",
    split="train",
    streaming=True,
)

streaming_dataset

IterableDataset({
    features: ['text'],
    num_shards: 1
})

С потоковыми данными нельзя работать как со списком. 

Такой датасет необходимо использовать как итератор

### 2.3 CommonCrawl

CommonCrawl - это большой архив веб-страниц. 

Неочищенный архив. Такого датасета хватит для предобучения LLM, но для начала нужно:
- удалить HTML-разметку и хэдеры
- удалить метаданные
- очистить датасет от мусора
- удалить неизвестные языки 
- удалить дубликаты
- удалить персональные данные
- удалить NSFW контент
- продолжение следует...

## 3. Токенизация

После подготовки текстовых данных, их необходимо перевести в числовые тензоры.

### 3.1 `AutoTokenizer`

В библиотеке `transformers` токенизатор обычно загружают следующим образом:

```python
tokenizer = AutoTokenizer.from_pretrained("...")
```

`AutoTokenizer` сам выбирает нужный класс токенизатора по имени модели

In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

text = """
There's a new kind of coding I call "vibe coding", where you fully give in to the vibes,
embrace exponentials, and forget that the code even exists.

(с) Andrej Karpathy
"""


encoded = tokenizer(
    text,
    return_tensors="pt",
)

print(encoded.input_ids)
print(tokenizer.decode(encoded.input_ids[0]))

tensor([[  198,  1858,   338,   257,   649,  1611,   286, 19617,   314,   869,
           366,    85, 32438, 19617,  1600,   810,   345,  3938,  1577,   287,
           284,   262, 16081,   274,    11,   198,   368, 46565,  1033,   261,
         14817,    11,   290,  6044,   326,   262,  2438,   772,  7160,    13,
           198,   198,     7, 21727,     8, 10948,    73,   509,  5117, 10036,
           198]])

There's a new kind of coding I call "vibe coding", where you fully give in to the vibes,
embrace exponentials, and forget that the code even exists.

(с) Andrej Karpathy



Посмотрим сами токены

In [14]:
ids = encoded.input_ids[0].tolist()
tokens = tokenizer.convert_ids_to_tokens(ids)

for token_id, token in zip(ids, tokens):
    print(f"{token_id:>6} -> {token}")

   198 -> Ċ
  1858 -> There
   338 -> 's
   257 -> Ġa
   649 -> Ġnew
  1611 -> Ġkind
   286 -> Ġof
 19617 -> Ġcoding
   314 -> ĠI
   869 -> Ġcall
   366 -> Ġ"
    85 -> v
 32438 -> ibe
 19617 -> Ġcoding
  1600 -> ",
   810 -> Ġwhere
   345 -> Ġyou
  3938 -> Ġfully
  1577 -> Ġgive
   287 -> Ġin
   284 -> Ġto
   262 -> Ġthe
 16081 -> Ġvib
   274 -> es
    11 -> ,
   198 -> Ċ
   368 -> em
 46565 -> brace
  1033 -> Ġexp
   261 -> on
 14817 -> entials
    11 -> ,
   290 -> Ġand
  6044 -> Ġforget
   326 -> Ġthat
   262 -> Ġthe
  2438 -> Ġcode
   772 -> Ġeven
  7160 -> Ġexists
    13 -> .
   198 -> Ċ
   198 -> Ċ
     7 -> (
 21727 -> Ñģ
     8 -> )
 10948 -> ĠAndre
    73 -> j
   509 -> ĠK
  5117 -> arp
 10036 -> athy
   198 -> Ċ


### 3.2 Параметры токенизатора

In [197]:
encoded = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=20,
    return_tensors="pt",
)

print("input_ids:")
print(encoded["input_ids"])

print("\npad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)

print("\neos_token:", tokenizer.eos_token)
print("eos_token_id:", tokenizer.eos_token_id)

print("\nbos_token:", tokenizer.bos_token)
print("bos_token_id:", tokenizer.bos_token_id)

print("\nattention_mask:")
print(encoded["attention_mask"])

print("\nvocab_size:", tokenizer.vocab_size)

print("\n Кодирование текста:")
token_ids = tokenizer.encode(texts[0])
print(token_ids)

print("\n Декодирование токенов в текст:")
decoded_text = tokenizer.decode(token_ids)
print(decoded_text)


input_ids:
tensor([[ 1858,   338,   257,   649,  1611,   286, 19617,   314,   869,   366,
            85, 32438, 19617,  1600,   810,   345,  3938,  1577,   287,   284],
        [  464,  2438,   318,   991,   612,    11,   475,   262,  1388,  7071,
          4329,  3288,  3303,    13, 50256, 50256, 50256, 50256, 50256, 50256]])

pad_token: <|endoftext|>
pad_token_id: 50256

eos_token: <|endoftext|>
eos_token_id: 50256

bos_token: <|endoftext|>
bos_token_id: 50256

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]])

vocab_size: 50257

 Кодирование текста:
[1858, 338, 257, 649, 1611, 286, 19617, 314, 869, 366, 85, 32438, 19617, 1600, 810, 345, 3938, 1577, 287, 284, 262, 16081, 274, 13]

 Декодирование токенов в текст:
There's a new kind of coding I call "vibe coding", where you fully give in to the vibes.


### 3.3 Токенизация датасета

In [15]:
raw_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

train_raw = raw_dataset["train"].filter(lambda example: len(example["text"].strip()) > 0)
val_raw = raw_dataset["validation"].filter(lambda example: len(example["text"].strip()) > 0)

train_raw = train_raw.select(range(2000))
val_raw = val_raw.select(range(300))

train_raw, val_raw

(Dataset({
     features: ['text'],
     num_rows: 2000
 }),
 Dataset({
     features: ['text'],
     num_rows: 300
 }))

In [16]:
def tokenize_batch(batch):
    return tokenizer(batch["text"])


tokenized_train = train_raw.map(
    tokenize_batch,
    batched=True,
    remove_columns=train_raw.column_names,
)

tokenized_val = val_raw.map(
    tokenize_batch,
    batched=True,
    remove_columns=val_raw.column_names,
)

tokenized_train

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 2000
})

### 3.4 Подготовка последовательностей одинаковой длины

После токенизации каждая строка превратилась в список `input_ids`. Но длины списков разные, поэтому для обучения мы нарежем общий поток токенов на куски одинаковой длины.

In [17]:
print(tokenized_train[0].keys())
print(tokenized_train[0]["input_ids"][:20])

dict_keys(['input_ids', 'attention_mask'])
[796, 569, 18354, 7496, 17740, 6711, 796, 220, 198]


In [18]:
block_size = 128


def group_texts(examples):
    all_input_ids = []

    for input_ids in examples["input_ids"]:
        all_input_ids.extend(input_ids)

    total_length = len(all_input_ids)
    total_length = (total_length // block_size) * block_size

    chunks = [
        all_input_ids[i : i + block_size]
        for i in range(0, total_length, block_size)
    ]

    return {
        "input_ids": chunks,
        "labels": [chunk.copy() for chunk in chunks],
    }


lm_train = tokenized_train.map(
    group_texts,
    batched=True,
    remove_columns=tokenized_train.column_names,
)
lm_val = tokenized_val.map(
    group_texts,
    batched=True,
    remove_columns=tokenized_val.column_names,
)

lm_train.set_format(type="torch")
lm_val.set_format(type="torch")

lm_train, lm_val

(Dataset({
     features: ['input_ids', 'labels'],
     num_rows: 1566
 }),
 Dataset({
     features: ['input_ids', 'labels'],
     num_rows: 201
 }))

In [19]:
example = lm_train[50]

print(example.keys())
print("input_ids shape:", example["input_ids"].shape)
print("labels shape:", example["labels"].shape)
print()
print(tokenizer.decode(example["input_ids"]))

dict_keys(['input_ids', 'labels'])
input_ids shape: torch.Size([128])
labels shape: torch.Size([128])

 arms , and machinery at the Little Rock Arsenal was removed to east of the Mississippi River by order of Maj. Gen. Earl Van Dorn in April and May 1862 , and accountability for it is lost at that point . By all appearances , the equipment was sent down the river to Napoleon , Arkansas , and from there to Jackson Mississippi , where it was probably destroyed during the Vicksburg campaign in the early summer of 1863 . 
 Major General Thomas C. Hindman , sent to command the district of Arkansas in May , 1862 , found the state nearly destitute of military material . Hindman established another armory at Arkadelphia , and


## 3.5 Данные

Создадим даталоадеры

In [20]:
train_loader = DataLoader(
    lm_train,
    batch_size=8,
    shuffle=True,
    collate_fn=default_data_collator,
)

val_loader = DataLoader(
    lm_val,
    batch_size=8,
    shuffle=False,
    collate_fn=default_data_collator,
)

batch = next(iter(train_loader))

print(batch.keys())
print("input_ids shape:", batch["input_ids"].shape)
print("labels shape:", batch["labels"].shape)

dict_keys(['input_ids', 'labels'])
input_ids shape: torch.Size([8, 128])
labels shape: torch.Size([8, 128])


## 4. Инфраструктура обучения

Для качественного обучения нам необходимо снимать метрики (чтобы оценивать процесс обучения), где то их логировать, следом отображать.

Полезным также будет отслеживать метрики в виде некоторой инфографики.

При обучении больших систем необходимо также замерять хардварные статистики: насколько хорошо утилизируется гпу, как сильно нагрелись процессоры, сколько оперативной памяти занято и тд.

В процессе обучения необходимо делать сохранение моделей в чекпоинты. чтобы, например, зафиксировать прогресс в обучении (мало ли оно упадет)

### 4.1. Логика обработки данных и обучения

Данные:

In [22]:
class WikiTextDataModule(LightningDataModule):
    def __init__(
        self,
        tokenizer_name: str = "gpt2",
        block_size: int = 128,
        batch_size: int = 8,
        train_size: int = 2000,
        val_size: int = 300,
    ):
        super().__init__()
        self.tokenizer_name = tokenizer_name
        self.block_size = block_size
        self.batch_size = batch_size
        self.train_size = train_size
        self.val_size = val_size

    def prepare_data(self):
        load_dataset("wikitext", "wikitext-2-raw-v1")
        AutoTokenizer.from_pretrained(self.tokenizer_name)

    def setup(self, stage=None):
        self.tokenizer = AutoTokenizer.from_pretrained(self.tokenizer_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        raw_dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

        train_raw = raw_dataset["train"].filter(lambda example: len(example["text"].strip()) > 0)
        val_raw = raw_dataset["validation"].filter(lambda example: len(example["text"].strip()) > 0)

        train_raw = train_raw.select(range(min(self.train_size, len(train_raw))))
        val_raw = val_raw.select(range(min(self.val_size, len(val_raw))))

        def tokenize_batch(batch):
            return self.tokenizer(batch["text"])

        tokenized_train = train_raw.map(
            tokenize_batch,
            batched=True,
            remove_columns=train_raw.column_names,
        )

        tokenized_val = val_raw.map(
            tokenize_batch,
            batched=True,
            remove_columns=val_raw.column_names,
        )

        def group_texts(examples):
            all_input_ids = []

            for input_ids in examples["input_ids"]:
                all_input_ids.extend(input_ids)

            total_length = len(all_input_ids)
            total_length = (total_length // self.block_size) * self.block_size

            chunks = [
                all_input_ids[i : i + self.block_size]
                for i in range(0, total_length, self.block_size)
            ]

            return {
                "input_ids": chunks,
                "labels": [chunk.copy() for chunk in chunks],
            }

        self.train_ds = tokenized_train.map(
            group_texts,
            batched=True,
            remove_columns=tokenized_train.column_names,
        )
        self.val_ds = tokenized_val.map(
            group_texts,
            batched=True,
            remove_columns=tokenized_val.column_names,
        )

        self.train_ds.set_format(type="torch")
        self.val_ds.set_format(type="torch")

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            shuffle=True,
            collate_fn=default_data_collator,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=default_data_collator,
        )

In [23]:
data_module = WikiTextDataModule(
    tokenizer_name="gpt2",
    block_size=128,
    batch_size=8,
    train_size=2000,
    val_size=300,
)

Модель:

In [24]:
class TinyLM(LightningModule):
    def __init__(
        self,
        vocab_size: int,
        block_size: int = 128,
        n_layer: int = 2,
        n_head: int = 2,
        n_embd: int = 128,
        lr: float = 3e-4,
        pad_token_id: int | None = None,
        bos_token_id: int | None = None,
        eos_token_id: int | None = None,
    ):
        super().__init__()
        self.save_hyperparameters()

        config = GPT2Config(
            vocab_size=vocab_size,
            n_positions=block_size,
            n_ctx=block_size,
            n_layer=n_layer,
            n_head=n_head,
            n_embd=n_embd,
            pad_token_id=pad_token_id,
            bos_token_id=bos_token_id,
            eos_token_id=eos_token_id,
        )

        self.model = GPT2LMHeadModel(config)

    def forward(self, input_ids, labels=None):
        return self.model(
            input_ids=input_ids,
            labels=labels,
        )

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)

    def training_step(self, batch, batch_idx):
        start_time = time.time()

        outputs = self(
            input_ids=batch["input_ids"],
            labels=batch["labels"],
        )

        loss = outputs.loss

        tokens = batch["input_ids"].numel()
        elapsed = max(time.time() - start_time, 1e-6)
        tokens_per_sec = tokens / elapsed

        self.log("train/loss", loss, on_step=True, on_epoch=False, prog_bar=True)
        self.log("speed/tokens_per_sec", tokens_per_sec, on_step=True, on_epoch=False, prog_bar=False)

        return loss

    def validation_step(self, batch, batch_idx):
        outputs = self(
            input_ids=batch["input_ids"],
            labels=batch["labels"],
        )

        loss = outputs.loss

        perplexity = torch.exp(torch.clamp(loss, max=20))

        self.log("val/loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log("val/perplexity", perplexity, on_step=False, on_epoch=True, prog_bar=True)

In [25]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

model = TinyLM(
    vocab_size=len(tokenizer),
    block_size=128,
    n_layer=2,
    n_head=2,
    n_embd=128,
    lr=3e-4,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

sum(p.numel() for p in model.parameters())

6846080

### 4.2 Чекпоинты

Чекпоинт хранит текущее состояние обучения: веса моделей, optimizer state, номер эпохи и текущий шаг обучения, состояния колбеков и тд

В lightning чекпоинты удобно сохранять через колбек `ModelCheckpoint` 

In [26]:
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints/tiny_lm",
    filename="tiny-lm-{epoch:02d}-{val_loss:.3f}",
    monitor="val/loss",
    mode="min",
    save_top_k=2,
    save_last=True,
)


Добавим еще и монитор learning rate

In [27]:
lr_monitor = LearningRateMonitor(logging_interval="step")

### 4.3 ClearML


In [28]:
USE_CLEARML = True

if USE_CLEARML:
    from clearml import Task
    import os

    os.makedirs("clearml_outputs", exist_ok=True)
    
    task = Task.init(
        project_name="seminars/llm",
        task_name="tiny-lm",
        output_uri="./clearml_outputs",
    )
else:
    task = None

ClearML Task: created new task id=b4b285ad56384498af24b47e306c8702
ClearML results page: https://app.clear.ml/projects/d2db9a8c14824c8392126fdf7ad948cc/experiments/b4b285ad56384498af24b47e306c8702/output/log
2026-04-25 16:44:07,208 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found


ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


In [211]:
# task.close()  # чтобы завершить задачу

Подключиться к ClearML: https://app.clear.ml

### 4.4 Тензорборд-логирование

In [29]:
logger = TensorBoardLogger(
    save_dir="logs",
    name="tiny_lm",
)

## 5. Обучим модель

In [30]:
trainer = Trainer(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    logger=logger,
    callbacks=[
        checkpoint_callback,
        lr_monitor,
    ],
    log_every_n_steps=10,
    limit_train_batches=100,
    limit_val_batches=20,
)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(model, datamodule=data_module)

Map: 100%|██████████| 300/300 [00:00<00:00, 21990.79 examples/s]


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ GPT2LMHeadModel │  6.8 M │ train │     0 │
└───┴───────┴─────────────────┴────────┴───────┴───────┘

Trainable params: 6.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 6.8 M                                                                                                
Total estimated model params size (MB): 27                                                                         
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

In [ ]:
print("best checkpoint:", checkpoint_callback.best_model_path)
print("last checkpoint:", checkpoint_callback.last_model_path)

best checkpoint: /Users/egorprokopov/Documents/Work/MI/MNNA-2026-Spring-Internal/Seminars/Sem2/checkpoints/tiny_lm/tiny-lm-epoch=00-val_loss=0.000-v3.ckpt
last checkpoint: /Users/egorprokopov/Documents/Work/MI/MNNA-2026-Spring-Internal/Seminars/Sem2/checkpoints/tiny_lm/last-v3.ckpt


Если обучение прервалось, его можно продолжить с последнего чекпоинта

In [216]:
# trainer.fit(
#     model,
#     datamodule=data_module,
#     ckpt_path="checkpoints/tiny_causal_lm/last.ckpt",
# )

Или можно загрузить модель из чекпоинта отдельно

In [217]:
best_checkpoint_path = checkpoint_callback.best_model_path

if best_checkpoint_path:
    loaded_model = TinyLM.load_from_checkpoint(best_checkpoint_path)
    loaded_model.eval()
else:
    loaded_model = model